# Customer Info Silver Layer
Welcome to the silver layer! The goals of this notebook is to take everything from the crm_cust_info bronze layer and clean it up for the gold layer.
<br></br>
The data may still be difficult to work with by the end but it will be ready for the final round of transformation.


In [0]:
%sql
-- First off lets get take a quick glance at whats going on in this dataset.
SELECT
    *
FROM workspace.`bike-bronze`.crm_cust_info
LIMIT 10;

In [0]:
%sql
-- Quickly checking the count of all rows to see if distnct ids & keys are matching the total amount of rows.
-- If not, we will need to explore why that is the case.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT cst_id) AS unique_ids,
    COUNT(distinct cst_key) AS cst_keys
FROM workspace.`bike-bronze`.crm_cust_info;

In [0]:
%sql
-- Dupes need to be removed, there exists 6 non-null IDs which have dupes.
-- This explains some of the issues we were seeing above but, where are the other 4?
SELECT 
    CST_ID,
    count(cst_id)
FROM workspace.`bike-bronze`.crm_cust_info
GROUP BY ALL
HAVING COUNT(cst_id) > 1;

In [0]:
%sql
-- Here we are selecting rows which are duplicates by partitioning by the customer ID and ordering by the create date.
-- It looks like we discovered 3 new problem rows by using this window function. It appears that null was not included in the previous pull.
-- This is because the COUNT function does not include null values. We see 3 cst_keys that are not matching what we expect.
-- There is likely a 4th but since we filter on row_num > 1 we will have to seek it out elsewhere.
WITH cte_dupes AS (
    SELECT *, ROW_NUMBER() OVER(
        PARTITION BY cst_id
        ORDER BY cst_create_date DESC
    ) AS row_num
     FROM workspace.`bike-bronze`.crm_cust_info
)
SELECT * FROM cte_dupes WHERE row_num > 1;

In [0]:
%sql
-- Some more exploration, 4K rows where customer gender is null out of 18494 total entries (with duplicates).
-- It is present in the other table for some values. Likely will need to impute.
SELECT
    'Rows with gender null' AS desc
    ,COUNT(*) AS count
FROM workspace.`bike-bronze`.crm_cust_info
where cst_gndr IS NULL
UNION ALL
SELECT
    'Total rows' AS desc
    ,COUNT(*) AS count
FROM workspace.`bike-bronze`.crm_cust_info;

In [0]:
%sql
-- Here we find the rest of the problem cst_keys, these will have to be removed as well.
-- They are completely useless for us as they do not have any information.
-- We can see that this is the reason why our initial exploration was not matching the count of unique cst_ids.
SELECT
    *
FROM workspace.`bike-bronze`.crm_cust_info
WHERE
    cst_key NOT LIKE '%AW%';

In [0]:
%sql
-- Values here are exactly what we expect. They come from the dupes that were present earlier.
SELECT
    *
FROM workspace.`bike-bronze`.crm_cust_info
WHERE
    cst_marital_status IS NULL;

In [0]:
%sql
-- One last check, making sure that we have a consistent structure for gender column.
SELECT
    DISTINCT
    cst_gndr
FROM workspace.`bike-bronze`.crm_cust_info;

# So lets recap whats going on here
We have a couple of issues we need to address from the bronze layer.
- There exists 6 duplicate rows for rows that have a cst_id.
- There are 4 rows with no data that have a cst_key that is in an unexpected format.
- First & last names need to be readjusted so that they do not have any trailing spaces.
- Gender values will be imputed from erp_customers table in the gold layer


In [0]:
df = spark.sql("SELECT * FROM workspace.`bike-bronze`.crm_cust_info")
df.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# We begin our transformations here by remaking the window function from above in pyspark
# The idea is the same, we break the data up by customer id then rank it based on the create date.

window_spec = Window.partitionBy(df["cst_id"]).orderBy("cst_create_date")

# We have some additional logic here because we know that our stragler is going to be a null value
# So we need to filter out the null values as well as the duplicates
# This isn't the best way of doing things, it's a little bit hard coded but it works for the circumstance.

df = df.withColumn("row_count",F.row_number().over(window_spec)) \
                    .filter((F.col("row_count") == 1) & (F.col("cst_id").isNotNull())) \
                    .drop("row_count")
# Check row count, should be 18484

df.count()

In [0]:
# Triming the trailing and leading spaces in the first and last name column
trailing_columns = ['cst_firstname', 'cst_lastname']

for i in trailing_columns:
  df = df.withColumn(i,F.trim(F.col(i)))

df.display()